# Portfolio construction

Sleeves (each a strategy's positions as if it had the whole book) are allocated, netted per instrument and run
through the engine as one book. Allocations refit monthly from past sleeve returns only.

Compare the methods, then look at what whole contracts and stress do to the chosen one.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quantlab.lab import Lab
from quantlab.signals import library as lib
from quantlab.reporting import charts

charts.apply_style()
pd.set_option("display.width", 200)

from quantlab.engine.backtest import Sizer, run_backtest
from quantlab.engine.components import Targets
from quantlab.analytics.performance import metrics
from quantlab.portfolio.construction import build_portfolio, to_contracts
from quantlab.validation import scenarios

lab = Lab(universe="equity_index_core", start="1999", show=False)
sleeves = {name: lab.backtest(name).decided for name in ["trend", "breakout", "momentum", "reversion"]}

## Allocation methods

In [ ]:
books = {}
for method in ["equal", "inverse_vol", "risk_parity", "hrp"]:
    weights = build_portfolio(sleeves, lab.returns, method=method, lookback=252, refit_every=21)
    books[method] = (weights, run_backtest(Targets(weights.weights), lab.returns, costs_bps=lab.costs, sizer=Sizer(target_vol=None), config=lab.engine))
pd.DataFrame({m: metrics(result) for m, (_w, result) in books.items()}).T[["Sharpe", "annual return", "volatility", "max drawdown", "turnover"]].round(3)

In [ ]:
chosen = "risk_parity"
weights, result = books[chosen]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
weights.scales.plot(ax=axes[0], title=f"{chosen}: sleeve scales over time")
charts.equity({m: r for m, (_w, r) in books.items()}, ax=axes[1])
plt.tight_layout(); plt.show()

## Whole contracts

At a given capital, how far are the held weights from the intended ones? Micro contracts on the US indices (others need FX; shown in their own currency).

In [ ]:
capital = 10_000_000 / 10.5      # 10 million NOK in USD, roughly
us = [name for name in lab.instruments if name in ("idx:SPX", "idx:NDX", "idx:RUT")]
multipliers = {name: lab._master[lab._master[name].proxy_for].multiplier for name in us}
contracts = to_contracts(weights.weights[us], equity=capital, prices=lab.panel["close"][us], multipliers=multipliers)
print("share of days a wanted position rounds to zero:", contracts["rounded_to_zero"].mean().round(3).to_dict())
(weights.weights[us] - contracts["held_weights"]).abs().mean().round(4)

## Stress

In [ ]:
display(scenarios.historical(result.pnl, benchmark=lab.returns.mean(axis=1)).round(3))
scenarios.stress(result.held, result.returns)["table"].round(3)